# Measuring the 2D power spectrum of an image — **JAX version**

Same nine-step walk-through as before, with the numerics written in JAX
(`jax.numpy`, `jax.random`, `jax.jit`), followed by a compact parameter study of
how the recovered $P(k)$ responds to the **box size $L$**, the **field grid
$N_{\rm grid}$**, and the **measurement grid $N_{\rm grid\_Pk\_2D}$**, one at a time.

**Estimator convention** (the project's 1D `dv/Npix`, promoted to 2D):

$$\tilde\delta(\vec k)=\mathrm{d}x^2\,\mathrm{FFT2}(\delta),\qquad
\hat P(\vec k)=\frac{|\tilde\delta(\vec k)|^2}{V}
=\frac{\mathrm{d}x^2}{N_{\rm pix}}\,|\mathrm{FFT2}(\delta)|^2,\quad V=L^2 .$$

*Note on the embedded figures: they were pre-rendered through a float64
NumPy-equivalent of this code (JAX was unavailable in the build sandbox).
Re-running under JAX gives the same systematics; the specific random
realization differs because JAX's PRNG ≠ NumPy's.*

In [ ]:
%matplotlib widget
from functools import partial
from dataclasses import dataclass

import jax
jax.config.update("jax_enable_x64", True)   # float64, to match the analytic check
import jax.numpy as jnp
import numpy as np                          # host side: binning, plotting
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


@dataclass
class Config:
    L: float = 1.0            # box size
    N_grid: int = 512         # field-generation grid
    N_grid_Pk_2D: int = 64   # FFT / measurement grid

    A: float = 1.0e-3         # P(k) = A (k/k0)^(ns-1)
    k0: float = 2.0 * np.pi
    ns: float = 1.          # (ns-1) = -2.5 -> red, visually rich
    seed: int = 42

    dk_lin: float = None
    k_threshold_lin: float = None
    n_log_bins: int = 18

cfg = Config()


def analytic_Pk(kmod, A, k0, ns):
    """P(|k|) = A (k/k0)^(ns-1), with P(0):=0.  Works on JAX or host arrays."""
    kmod = jnp.asarray(kmod)
    safe = jnp.where(kmod > 0, kmod, 1.0)
    P = A * (safe / k0) ** (ns - 1.0)
    return jnp.where(kmod > 0, P, 0.0)


def fourier_grid(N, L):
    """Angular wavenumbers k = 2*pi*f for an N x N grid (fftfreq order)."""
    k1d = 2.0 * jnp.pi * jnp.fft.fftfreq(N, d=L / N)
    kx, ky = jnp.meshgrid(k1d, k1d, indexing="ij")
    return kx, ky, jnp.sqrt(kx**2 + ky**2)


# ---- host-side helpers (binning lives on the CPU, not inside jit) ----------
def make_hybrid_bin_edges(L, M, dk_lin=None, k_threshold_lin=None, n_log_bins=18):
    k_fund = 2.0 * np.pi / L              # smallest non-zero |k|
    k_nyq = np.pi * M / L                 # Nyquist of the measurement grid
    dk = dk_lin if dk_lin is not None else k_fund
    k_thr = k_threshold_lin if k_threshold_lin is not None else 12.0 * k_fund
    k_thr = min(k_thr, k_nyq)
    lin = np.arange(k_fund * 0.5, k_thr + dk, dk)
    log = np.geomspace(lin[-1], k_nyq, n_log_bins + 1)
    return np.concatenate([lin[:-1], log]), k_fund, k_nyq


def radial_average(P2d, kmod, edges, k_cent_mode='weighted'):
    """
    Computes the radial average of a 2D power spectrum.
    
    Parameters:
    -----------
    P2d : 2D array
        The 2D power spectrum.
    kmod : 2D array
        The 2D array of k magnitudes (wavenumbers).
    edges : 1D array
        The bin edges for the radial bins.
    k_cent_mode : str, optional
        Method to compute the central k value for each bin. Options are:
        - 'mean'    : Mean of the k values in the bin (default, original behavior).
        - 'linear'  : Central value of the bin in linear scale.
        - 'log'     : Central value of the bin in log scale (geometric mean).
        - 'weighted': Mean of k values weighted by the power spectrum P.
        
    Returns:
    --------
    k_cent : 1D array
        The central k value for each bin.
    P_mean : 1D array
        The mean power spectrum value for each bin.
    counts : 1D array
        The number of pixels/modes in each bin.
    """
    k, P = np.asarray(kmod).ravel(), np.asarray(P2d).ravel()
    
    # Filter out k=0 to avoid issues with log scales or undefined directions
    keep = k > 0
    k, P = k[keep], P[keep]
    
    idx = np.digitize(k, edges) - 1
    nbin = len(edges) - 1
    
    k_cent = np.full(nbin, np.nan)
    P_mean = np.full(nbin, np.nan)
    counts = np.zeros(nbin, int)
    
    valid_modes = ['mean', 'linear', 'log', 'weighted']
    if k_cent_mode not in valid_modes:
        raise ValueError(f"Invalid k_cent_mode. Expected one of {valid_modes}, got '{k_cent_mode}'")
        
    for b in range(nbin):
        m = idx == b
        counts[b] = m.sum()
        
        if counts[b] > 0:
            # Calculate k_cent based on the selected mode
            if k_cent_mode == 'mean':
                k_cent[b] = k[m].mean()
            elif k_cent_mode == 'linear':
                k_cent[b] = (edges[b] + edges[b+1]) / 2.0
            elif k_cent_mode == 'log':
                # Geometric mean is the exact midpoint in log space
                k_cent[b] = np.sqrt(edges[b] * edges[b+1]) 
            elif k_cent_mode == 'weighted':
                k_cent[b] = np.average(k[m], weights=P[m])
                
            # P_mean is always the simple arithmetic mean
            P_mean[b] = P[m].mean()
            
    return k_cent, P_mean, counts

## Step 1 — generate the GRF (jitted)

Filter unit-variance white noise so the modes obey
$\langle|\tilde\delta|^2\rangle = V\,P(k)$. `static_argnames=("N",)` keeps the
grid size compile-time constant; the PRNG key is the only dynamic input besides
the spectrum parameters.

In [ ]:
@partial(jax.jit, static_argnames=("N",))
def generate_grf(key, N, L, A, k0, ns):
    dx = L / N
    _, _, kmod = fourier_grid(N, L)
    P = analytic_Pk(kmod, A, k0, ns)
    white_k = jnp.fft.fft2(jax.random.normal(key, (N, N)))
    delta = jnp.fft.ifft2(white_k * jnp.sqrt(P) / dx).real
    return delta

key = jax.random.PRNGKey(cfg.seed)
delta = generate_grf(key, cfg.N_grid, cfg.L, cfg.A, cfg.k0, cfg.ns)
x_field = (np.arange(cfg.N_grid) + 0.5) * cfg.L / cfg.N_grid
print("field shape:", delta.shape, " | variance:", f"{float(delta.var()):.4e}")

## Step 2 — show the field realization

In [ ]:
def plot_field(field, x, title):
    field = np.asarray(field)
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    vmax = np.percentile(np.abs(field), 99)
    im = ax.imshow(field.T, origin="lower", cmap="RdBu_r", extent=[x[0], x[-1], x[0], x[-1]], vmin=-vmax, vmax=vmax)
    ax.set(xlabel="x", ylabel="y", title=title)
    fig.colorbar(im, ax=ax, label=r"$\delta(\vec{x})$")
    return fig

plot_field(delta, x_field, f"GRF on field grid ({cfg.N_grid}\u00b2, seed={cfg.seed})")
plt.show()

## Step 3 — resample onto the measurement grid (jitted)

**Anti-aliased** resampling by Fourier-mode truncation: FFT the field, keep only
the modes below the measurement grid's Nyquist $k_{\rm Nyq}=\pi M/L$
(zero-pad instead when the measurement grid is *finer*), then transform back.
This band-limits the field **before** sampling, so down-sampling can no longer
alias small-scale power into the measured band.

> **Why this replaced bilinear interpolation.** The original Step 3 used periodic
> bilinear interpolation, which *point-samples* the field. When $M<N$ that folds
> every mode above $k_{\rm Nyq}=\pi M/L$ back into the band; for the (flat)
> default spectrum it inflates the **whole** measured $P(k)$ by a constant
> $\sim (N/M)^2$. That is exactly the offset seen in the parameter-study panels
> ($N/M=2\Rightarrow{\times}4$, $N/M=4\Rightarrow{\times}16$). Truncating the
> Fourier modes removes the aliased power and the bias goes away.

In [ ]:
@partial(jax.jit, static_argnames=("M",))
def interpolate_to_pk_grid(delta_field, M):
    """Resample the field from its N x N grid onto the M x M measurement grid by
    Fourier-mode truncation / zero-padding -- an *anti-aliased* resample.

    Why not bilinear?  Bilinear interpolation point-samples the field, so when
    M < N every Fourier mode above the measurement Nyquist (k_Nyq = pi*M/L) is
    *aliased* back into the measured band.  For a (near-)white spectrum that
    inflates every measured mode by ~(N/M)**2 -- a flat, k-independent high bias.
    Cropping the Fourier modes band-limits the field to the smaller grid's
    Nyquist *before* sampling, so no high-k power folds in.  M == N is an exact
    copy; M > N zero-pads (adds no spurious power)."""
    N = delta_field.shape[0]
    if M == N:
        return delta_field
    F = jnp.fft.fftshift(jnp.fft.fft2(delta_field))   # zero-frequency centered
    K = min(N, M)
    sN, sM = (N - K) // 2, (M - K) // 2
    Fm = jnp.zeros((M, M), dtype=F.dtype)
    Fm = Fm.at[sM:sM + K, sM:sM + K].set(F[sN:sN + K, sN:sN + K])
    return jnp.fft.ifft2(jnp.fft.ifftshift(Fm)).real * (M / N) ** 2  # keep amplitude

delta_pk = interpolate_to_pk_grid(delta, cfg.N_grid_Pk_2D)
x_pk = (np.arange(cfg.N_grid_Pk_2D) + 0.5) * cfg.L / cfg.N_grid_Pk_2D
print("Pk-grid shape:", delta_pk.shape)

## Step 4 — show the interpolated field

In [ ]:
plot_field(delta_pk, x_pk, f"Interpolated onto Pk grid ({cfg.N_grid_Pk_2D}\u00b2)")
plt.show()

## Step 5 — 2D periodogram with the correct prefactors (jitted)

$\tilde\delta=\mathrm{d}x^2\,\mathrm{FFT2}(\delta)$, then
$\hat P=|\tilde\delta|^2/V=(\mathrm{d}x^2/N_{\rm pix})\,|\mathrm{FFT2}|^2$.

In [ ]:
@jax.jit
def power_spectrum_2d(delta_pk, L):
    M = delta_pk.shape[0]
    dx, V = L / M, L**2
    delta_tilde = dx**2 * jnp.fft.fft2(delta_pk)   # discrete continuous-FT estimate
    P2d = jnp.abs(delta_tilde) ** 2 / V            # = (dx^2 / N_pix) |FFT|^2
    kx, ky, kmod = fourier_grid(M, L)
    return P2d, kx, ky, kmod

P2d, kx, ky, kmod = power_spectrum_2d(delta_pk, cfg.L)
print("P2d shape:", P2d.shape)

## Step 6 — show the 2D $P(k_x,k_y)$

In [ ]:
def plot_pk_2d(P2d, kx, ky, rings=None):
    P = np.fft.fftshift(np.asarray(P2d))
    
    kx_arr = np.asarray(kx)
    ky_arr = np.asarray(ky)
    
    # If kx and ky are 1D, create meshgrids.
    if kx_arr.ndim == 1 and ky_arr.ndim == 1:
        kx_arr, ky_arr = np.meshgrid(kx_arr, ky_arr, indexing='ij')
        
    kx_shifted = np.fft.fftshift(kx_arr)
    ky_shifted = np.fft.fftshift(ky_arr)
    
    # Flatten ALL arrays to 1D for discrete scatter plotting
    KX = kx_shifted.flatten()
    KY = ky_shifted.flatten()
    P_flat = P.flatten()  # <--- FLATTEN P HERE
    
    # Calculate extent for setting axis limits
    if kx_arr.ndim == 2:
        kxs = np.fft.fftshift(kx_arr[:, 0])
        kys = np.fft.fftshift(ky_arr[0, :])
    else:
        kxs = np.fft.fftshift(kx_arr)
        kys = np.fft.fftshift(ky_arr)
    ext = [kxs[0], kxs[-1], kys[0], kys[-1]]

    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    
    # Filter out zero or negative values for LogNorm
    pos = P_flat[P_flat > 0]
    if len(pos) > 0:
        vmin, vmax = pos.min(), pos.max()
    else:
        vmin, vmax = 1e-10, 1.0 # Fallback if no positive values
        
    # Create mask on the FLATTENED P array!
    mask = P_flat > 0       # <--- MASK IS NOW 1D
    KX_plot = KX[mask]
    KY_plot = KY[mask]
    P_plot = P_flat[mask]   # <--- USE FLATTENED P
    
    # Plot discrete scatter points
    sc = ax.scatter(KX_plot, KY_plot, c=P_plot, cmap="magma", norm=LogNorm(vmin=vmin, vmax=vmax), s=20, edgecolors='none')
    
    ttl = "2D power spectrum" + ("  +  hybrid bin rings" if rings is not None else "")
    ax.set(xlabel=r"$k_x$", ylabel=r"$k_y$", title=ttl)
    fig.colorbar(sc, ax=ax, label=r"$P(k_x,k_y)$")
    
    # Set limits to match the original extent
    ax.set_xlim(ext[0], ext[1])
    ax.set_ylim(ext[2], ext[3])
    
    if rings is not None:
        for r in rings:
            ax.add_patch(plt.Circle((0, 0), r, fill=False, color="cyan", lw=0.7, alpha=0.8))
                                    
    return fig

plot_pk_2d(P2d, kx, ky)
plt.show()

## Step 7 — hybrid (linear + log) radial bins

Linear edges up to `k_threshold_lin`, logarithmic edges out to Nyquist.

In [ ]:
edges, k_fund, k_nyq = make_hybrid_bin_edges(cfg.L, cfg.N_grid_Pk_2D)
print(f"{len(edges)-1} bins | k_fund={k_fund:.2f} | k_Nyq={k_nyq:.1f}")
print("first edges:", np.round(edges[:6], 2), "...")

## Step 8 — overlay the bin rings on the 2D map

In [ ]:
plot_pk_2d(P2d, kx, ky, rings=edges)
plt.show()

## Step 9 — radially average and compare to the input

In [ ]:
k_cent, P_meas, counts = radial_average(P2d, kmod, edges)

good = np.isfinite(k_cent) & (P_meas > 0)
kc, Pm = k_cent[good], P_meas[good]
kk = np.geomspace(kc.min(), kc.max(), 300)

fig, ax = plt.subplots(figsize=(6.2, 4.8))
ax.loglog(kk, np.asarray(analytic_Pk(kk, cfg.A, cfg.k0, cfg.ns)), "k-", lw=2, label="analytic input")
ax.loglog(kc, Pm, "o", ms=5, color="crimson", label="measured (binned)")
ax.axvline(0.8 * k_nyq, ls=":", color="gray", label=r"$0.8\,k_{\rm Nyq}$")
ax.set(xlabel=r"$|k|$", ylabel=r"$P(|k|)$", title="Input vs. measured power spectrum")
ax.legend(frameon=False)
plt.show()

band = good & (k_cent > 3 * k_fund) & (k_cent < 0.3 * k_nyq)
ratio = np.asarray(P_meas[band]) / np.asarray(analytic_Pk(k_cent[band], cfg.A, cfg.k0, cfg.ns))
print(f"measured / analytic (mid-band median): {np.median(ratio):.3f} (1.0 == unbiased prefactor)")

---
# Parameter study: $L$, $N_{\rm grid}$, $N_{\rm grid\_Pk\_2D}$ one at a time

The helper below reuses the jitted pipeline (steps 1, 3, 5) and **averages the
periodogram over a few PRNG keys** before binning, so each panel shows the
*systematic* response rather than per-realization cosmic-variance scatter.

What to expect:

* **Box size $L$** sets the accessible scales — fundamental $k_{\rm f}=2\pi/L$
  (low end) and Nyquist $k_{\rm Nyq}=\pi N/L$ (high end) — but **not** the
  amplitude: the $1/V$ prefactor removes box dependence, so all $L$ land on the
  *same* analytic curve.
* **$N_{\rm grid}$** (field grid, with the measurement grid fixed) sets the
  highest wavenumber that carries real signal. If $N_{\rm grid}<N_{\rm grid\_Pk\_2D}$
  the measurement cannot recover power the field never sampled (early cutoff);
  if $N_{\rm grid}>N_{\rm grid\_Pk\_2D}$ the field is *anti-alias* down-sampled
  (band-limited) to the measurement Nyquist, so the curve still lands on the
  analytic $P(k)$ — no high-$k$ power is folded back in.
* **$N_{\rm grid\_Pk\_2D}$** (measurement grid, with a well-resolved field) sets
  how far in $k$ you can measure, since $k_{\rm Nyq}=\pi M/L$; larger $M$ cleanly
  extends the reach up to the field's own Nyquist.

In [ ]:
def measure_avg_pk(L, N, M, seed=42, n_seeds=6, A=None, k0=None, ns=None):
    """Seed-averaged binned P(|k|) for a given (L, N_grid, N_grid_Pk_2D)."""
    A = cfg.A if A is None else A
    k0 = cfg.k0 if k0 is None else k0
    ns = cfg.ns if ns is None else ns
    edges, k_fund, k_nyq = make_hybrid_bin_edges(L, M)
    runs = []
    for key in jax.random.split(jax.random.PRNGKey(seed), n_seeds):
        d = generate_grf(key, N, L, A, k0, ns)
        dpk = interpolate_to_pk_grid(d, M)
        P2d_, _, _, kmod_ = power_spectrum_2d(dpk, L)
        kcent, Pm, _ = radial_average(P2d_, kmod_, edges)
        runs.append(Pm)
    return kcent, np.nanmean(np.vstack(runs), axis=0), k_fund, k_nyq


def run_panel(ax_main, ax_res, title, cases):
    """cases: list of (label, L, N, M).  Returns list of (label, mid-band bias)."""
    series, allk, biases = [], [], []
    for label, L, N, M in cases:
        kcent, Pm, kf, kn = measure_avg_pk(L, N, M)
        g = np.isfinite(kcent) & (Pm > 0)
        kg, Pg = kcent[g], Pm[g]
        series.append((label, kg, Pg))
        allk += [kg.min(), kg.max()]
        
        # Calculate mid-band bias
        b = (kg > 3 * kf) & (kg < 0.3 * kn)
        ana_b = np.asarray(analytic_Pk(kg[b], cfg.A, cfg.k0, cfg.ns))
        biases.append((label, float(np.median(Pg[b] / ana_b))))
        
        # Calculate residuals (ratio) for the whole range and plot them
        ana_full = np.asarray(analytic_Pk(kg, cfg.A, cfg.k0, cfg.ns))
        ratio = Pg / ana_full
        ax_res.plot(kg, ratio, 'o', ms=4, label=label)
        
    # Plot analytical line on the main axis
    kk = np.geomspace(min(allk), max(allk), 400)
    ax_main.loglog(kk, np.asarray(analytic_Pk(kk, cfg.A, cfg.k0, cfg.ns)), "k-", lw=1.8, label="analytic", zorder=1)
              
    # Plot measured points on the main axis
    for label, kg, Pg in series:
        ax_main.loglog(kg, Pg, "o", ms=4, label=label)
        
    ax_main.set(title=title)
    ax_main.legend(fontsize=8, frameon=False)
    
    # Configure the residuals axis
    ax_res.axhline(1.0, color='k', ls='--', lw=1) # Perfect agreement line
    ax_res.set_xscale('log')
    ax_res.set_ylabel(r"$P_{\mathrm{meas}}/P_{\mathrm{ana}}$")
    ax_res.margins(y=0.1) # Add a little vertical padding
    
    return biases


# Create a 2x3 grid of axes, sharing the x-axis height_ratios gives more vertical space to the main plots (3 parts top, 1 part bottom)
fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

axes[0, 0].set_ylabel(r"$P(|k|)$")

# Remove x-tick labels from the top row since they are shared with the bottom row
for ax in axes[0]:
    ax.tick_params(labelbottom=False)

# Run the panels
bias_L = run_panel(axes[0, 0], axes[1, 0], "vary box size L   (N=256, M=256)", [(f"L={L}", L, 256, 256) for L in (0.1, 0.5, 1.0, 2.0, 4.0)])
bias_N = run_panel(axes[0, 1], axes[1, 1], "vary N_grid   (L=1, M=256)", [(f"N={N}", 1.0, N, 256) for N in (64, 128, 256, 512, 1024)])
bias_M = run_panel(axes[0, 2], axes[1, 2], "vary N_grid_Pk_2D   (L=1, N=256)", [(f"M={M}", 1.0, 256, M) for M in (64, 128, 256, 512, 1024)])

# Set x-label for the bottom row
for ax in axes[1]:
    ax.set_xlabel(r"$|k|$")

plt.tight_layout()
plt.show()